# CPU：全量 EEG / MEG 缓存预热
选择纯 CPU 运行时，依次执行。源码只从 GitHub 获取，不嵌入源码包。
原始数据保留在 Drive；本流程调用同一套 `Experiment.data.build()`，不创建神经网络或 GPU Trainer。
优先读取 `data/SpanishBCBL`，同时兼容 `data/SpanishBCBL_Full`。
CPU/GPU 统一使用本地 `/content/studies_links` 软链接与 `/content/local_cache` 缓存，避免在 Drive 创建软链接或写入 exca 缓存。
使用「全部运行」即可依次完成环境安装、缓存预热和归档；耗时步骤会实时显示日志与报错。
首次全量预处理可能很久；当前运行时内重跑会复用已完成的缓存。预热完成后执行归档单元，将缓存持久保存到 Drive。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive


def run_live(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


drive.mount("/content/drive")
REPO_URL = "https://github.com/wdongdong0708-jpg/brain2qwertyv1Test.git"
CODE = Path("/content/code")
WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
if (CODE / ".git").exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
os.chdir(CODE)
for folder in ("data", "archives", "runs"):
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)
raw_candidates = [WORKSPACE / "data/SpanishBCBL", WORKSPACE / "data/SpanishBCBL_Full"]
raw = next((path for path in raw_candidates if path.is_dir()), None)
if raw is None:
    raise FileNotFoundError(f"SpanishBCBL data directory not found; checked: {raw_candidates}")
raw = raw.resolve()
STUDIES_LINKS = Path("/content/studies_links")
STUDIES_LINKS.mkdir(parents=True, exist_ok=True)
for name in ("Pinet2024Eeg", "Pinet2024Meg"):
    alias = STUDIES_LINKS / name
    if alias.is_symlink() and alias.resolve() != raw:
        alias.unlink()
    if not alias.is_symlink():
        if alias.exists():
            raise FileExistsError(f"Expected a local symlink, but found an existing directory/file: {alias}")
        alias.symlink_to(raw, target_is_directory=True)
os.environ["BRAIN2QWERTY_STUDIES"] = str(STUDIES_LINKS)
os.environ["BRAIN2QWERTY_CACHE"] = "/content/local_cache"
Path(os.environ["BRAIN2QWERTY_CACHE"]).mkdir(parents=True, exist_ok=True)
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")
print(f"Raw data: {raw}; study links: {STUDIES_LINKS}; cache: /content/local_cache")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
run_live([sys.executable, "-u", "-m", "scripts.colab_setup"])
PYTHON = "/content/b2q-venv/bin/python"


In [ ]:
MODALITIES = ["eeg", "meg"]
CPU_THREADS = 2
for modality in MODALITIES:
    run_live([PYTHON, "-u", "-m", "brain2qwerty_v1", "cache",
              "--modality", modality, "--cpu-threads", str(CPU_THREADS)])


所有缓存完成后执行：将 `/content/local_cache` 打包到 Drive 的 `archives/features.tar`。GPU 实例解压到同一路径后训练。Colab 本地缓存随运行时释放，需完成归档才能持久保存。打包时不要同时启动写入这些缓存的任务。


In [ ]:
run_live([PYTHON, "-u", "-m", "scripts.cache_archive", "pack",
          "--cache-root", os.environ["BRAIN2QWERTY_CACHE"],
          "--archive", str(WORKSPACE / "archives/features.tar")])
